# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [9]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [10]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

In [11]:
  import zipfile
  zipfile.ZipFile("pasos_01.zip").extractall(".")

## 1. Carga de artefactos

In [12]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [13]:
feature_columns = contract["features"]

prep_tree = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), feature_columns),
])

prep_scaled = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
    ]), feature_columns),
])

**Respuesta:** Si el `StandardScaler` se ajusta sobre todo el conjunto antes del CV, cada fold de validación ya habría influido en la media y la desviación estándar que usa el scaler. Eso filtra información de validación hacia el entrenamiento y sobreestima el desempeño. Dentro del `Pipeline`, el scaler se ajusta solo con los datos de entrenamiento de cada fold.

## 3. Modelos candidatos

In [14]:
models = {
    "decision_tree": Pipeline([
        ("prep", prep_tree),
        ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "logistic_regression": Pipeline([
        ("prep", prep_scaled),
        ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]),
    "knn": Pipeline([
        ("prep", prep_scaled),
        ("clf", KNeighborsClassifier()),
    ]),
}

## 4. Comparación inicial con validación cruzada

In [15]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, pipe in models.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring="accuracy", return_train_score=True)
    rows.append({
        "model": name,
        "train_acc_mean": res["train_score"].mean(),
        "val_acc_mean": res["test_score"].mean(),
        "val_acc_std": res["test_score"].std(),
    })
initial_comparison = pd.DataFrame(rows).sort_values("val_acc_mean", ascending=False)
print(initial_comparison.round(4).to_string(index=False))

              model  train_acc_mean  val_acc_mean  val_acc_std
logistic_regression          1.0000        0.9791       0.0277
                knn          0.9771        0.9507       0.0353
      decision_tree          1.0000        0.9022       0.0586


## 5. Optimización de hiperparámetros

In [16]:
param_grids = {
    "decision_tree": {"clf__max_depth": [2, 3, 4, 5, None], "clf__min_samples_leaf": [1, 2, 5]},
    "logistic_regression": {"clf__C": [0.01, 0.1, 1, 10, 100]},
    "knn": {"clf__n_neighbors": [3, 5, 7, 9, 11], "clf__weights": ["uniform", "distance"]},
}

searches = {}
for name, pipe in models.items():
    gs = GridSearchCV(pipe, param_grids[name], cv=cv, scoring="accuracy", n_jobs=1)
    gs.fit(X_train, y_train)
    searches[name] = gs

tuned_comparison = pd.DataFrame([
    {"model": n, "best_cv_accuracy": gs.best_score_, "best_params": gs.best_params_}
    for n, gs in searches.items()
]).sort_values("best_cv_accuracy", ascending=False)
print(tuned_comparison.to_string(index=False))

              model  best_cv_accuracy                                        best_params
logistic_regression          0.979064                                      {'clf__C': 1}
                knn          0.957882 {'clf__n_neighbors': 7, 'clf__weights': 'uniform'}
      decision_tree          0.902463  {'clf__max_depth': 5, 'clf__min_samples_leaf': 2}


## 6. Selección y persistencia del campeón

In [17]:
champion_name = tuned_comparison.iloc[0]["model"]
champion = searches[champion_name].best_estimator_
print("Campeón:", champion_name, "| CV accuracy:", round(searches[champion_name].best_score_, 4))

joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

cv_results = pd.DataFrame(searches[champion_name].cv_results_)
cv_results.to_csv(REPORTS_DIR / "cv_results.csv", index=False)

training_metadata = {
    "champion": champion_name,
    "best_params": searches[champion_name].best_params_,
    "cv_accuracy": float(searches[champion_name].best_score_),
    "cv": "StratifiedKFold(n_splits=5, shuffle=True, random_state=42)",
    "features": feature_columns,
    "target": contract["target"],
    "tuned_summary": tuned_comparison.to_dict(orient="records"),
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("Artefactos guardados.")

Campeón: logistic_regression | CV accuracy: 0.9791
Artefactos guardados.


In [18]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

Elegí **regresión logística** (C = 1), con CV accuracy ≈ 0.979. Tiene la mejor validación cruzada y la segunda menor desviación estándar (≈ 0.028), frente a KNN (≈ 0.951 ± 0.035) y el árbol (≈ 0.902 ± 0.059). El árbol muestra sobreajuste claro: 1.0 en train contra 0.90 en validación, incluso después de limitar la profundidad. KNN es competitivo, pero depende de la escala y es menos interpretable. La logística es un modelo lineal simple que se puede explicar por coeficientes. Con 142 filas, esa simplicidad pesa a favor.
